# DPAugmentor Tutorial

`DPAugmentor` applies randomized augmentations to electron diffraction patterns for ML training data generation. Each call to `augment()` samples fresh parameter values and applies the full pipeline.

**Pipeline order (left → right):**  
`flipshift` → `shift / ellipticity / scale` → `inelastic background` → `shot noise` → `Gaussian noise` → `blur` → `salt & pepper`

**Parameter ranges:** Every numeric parameter accepts a fixed value, a `[min, max]` range (uniform sampling), or a weighted mixture of ranges — see Section 4.

Nicholas Marchese | Last updated: June 2026

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import numpy as np
import quantem as em
from quantem.core import config
from quantem.core.utils.augment_dp import DPAugmentor
from quantem.core.visualization import show_2d

GPU_IND = 0
config.set({"device": f"cuda:{GPU_IND}"})

In [ ]:
# Load real data if available; otherwise fall back to a synthetic ring pattern.
f_data = Path("~/data/example_data/polycrystal_2D_WS2.h5").expanduser()

if f_data.exists():
    dset = em.io.read_emdfile_to_4dstem(
        str(f_data),
        data_keys=["4DSTEM", "datacube", "data"],
        calibration_keys=["4DSTEM", "datacube", "calibration"],
    )
    dset.units = ["pixels"] * 4
    dset.sampling = [1] * 4
    dp = dset[0, 0].array.astype(np.float32)
    dp_stack = dset[::32, 64].array.astype(np.float32)
    print(f"Loaded real data — single dp: {dp.shape}, stack: {dp_stack.shape}")
else:
    H, W = 128, 128
    yy, xx = np.mgrid[-H//2:H//2, -W//2:W//2].astype(np.float32)
    r = np.sqrt(xx**2 + yy**2)
    dp = np.zeros((H, W), dtype=np.float32)
    for r0, width, amp in [(20, 3, 1.0), (38, 2.5, 0.6), (52, 2.0, 0.35)]:
        dp += amp * np.exp(-0.5 * ((r - r0) / width) ** 2)
    _rng0 = np.random.RandomState(42)
    dp += _rng0.exponential(0.02, (H, W)).astype(np.float32)
    dp = dp.clip(0)
    dp_stack = np.stack([dp * _rng0.uniform(0.8, 1.2) for _ in range(8)])
    print(f"Using synthetic ring pattern — single dp: {dp.shape}, stack: {dp_stack.shape}")

# Gaussian approximation of a focused electron probe
H, W = dp.shape
yy, xx = np.mgrid[-H//2:H//2, -W//2:W//2].astype(np.float32)
probe = np.exp(-0.5 * (xx**2 + yy**2) / 8.0**2).astype(np.float32)
probe /= probe.sum()

show_2d([dp, probe], title=["Input diffraction pattern", "Probe function"],
        upper_quantile=0.99, cbar=True)

## 1. Geometric Transforms

Geometric augmentations warp the diffraction pattern in real space and are applied first in the pipeline, so all subsequent noise acts on the already-distorted pattern.

### 1a. Flips & Rotation (`add_flipshift`)

Randomly flips horizontally and/or vertically (each with 50% probability) and applies a rotation.

- `free_rotation=False` (default): rotation locked to 0°, 90°, 180°, 270°  
- `free_rotation=True`: rotation drawn uniformly from `rotation_range`

In [ ]:
aug_90 = DPAugmentor(add_flipshift=True, free_rotation=False, rng=0)
dp_90 = aug_90.augment(dp)
aug_90.print_params()

aug_free = DPAugmentor(add_flipshift=True, free_rotation=True,
                       rotation_range=[-180, 180], rng=1)
dp_free = aug_free.augment(dp)
print(f"Free rotation drew: {aug_free.rotation_angle:.1f}°")

show_2d([dp, dp_90, dp_free],
        title=["Original", "90° rotation", "Free rotation"],
        upper_quantile=0.99, cbar=True)

### 1b. Shift (`add_shift`)

Translates the pattern by a random amount. The magnitude is sampled from `[0, xshift]` / `[0, yshift]` and a random sign is applied independently to each axis.

### 1c. Scale (`add_scale`)

Uniformly scales the pattern by `scale_factor`. Values > 1 zoom out; values < 1 zoom in. Set the minimum to 1.0 to only magnify.

> **Note:** Large shifts, free rotations, and scale < 1 leave zero-padded regions at the image edges. Use `scale_factor ≥ 1` and moderate ellipticity to minimise this.

In [ ]:
aug_shift = DPAugmentor(add_shift=True, xshift=[0, 20], yshift=[0, 20], rng=2)
dp_shift = aug_shift.augment(dp)
print(f"Shift drawn: x={aug_shift.xshift:.1f}, y={aug_shift.yshift:.1f} px")

aug_scale = DPAugmentor(add_scale=True, scale_factor=[0.8, 1.2], rng=3)
dp_scale = aug_scale.augment(dp)
print(f"Scale factor drawn: {aug_scale.scale_factor:.3f}")

aug_both = DPAugmentor(add_shift=True, xshift=[0, 15], yshift=[0, 15],
                       add_scale=True, scale_factor=[0.9, 1.1], rng=4)
dp_both = aug_both.augment(dp)

show_2d([dp, dp_shift, dp_scale, dp_both],
        title=["Original", "Shift only", "Scale only", "Shift + Scale"],
        upper_quantile=0.99, cbar=True)

### 1d. Ellipticity (`add_ellipticity`)

Applies an anisotropic linear distortion parameterized by strain tensor components `exx`, `eyy`, `exy`. These are drawn from a normal distribution with `ellipticity_scale` as σ, then normalized so `(exx + eyy) / 2 = 1` (area-preserving on average).

The same distortion is applied to the inelastic background (Section 2a): when `add_ellipticity=True`, the plasmon form factor `1/(q² + q₀²)` is evaluated in elliptically-transformed q-space so the background matches the distorted pattern geometry.

- `add_ellipticity_to_label=True` (default): label is warped identically to the DP  
- `add_ellipticity_to_label=False`: label only gets shift/scale/flip — useful when training a network to predict and correct ellipticity

In [ ]:
label_1ch = (dp > dp.mean()).astype(np.float32)

# Strong ellipticity so the effect is clearly visible
aug_ell_on = DPAugmentor(add_ellipticity=True, ellipticity_scale=0.12,
                          add_ellipticity_to_label=True, rng=5)
dp_ell_on, label_ell_on = aug_ell_on.augment(dp, label=label_1ch)
print(f"exx={aug_ell_on.exx:.3f}, eyy={aug_ell_on.eyy:.3f}, exy={aug_ell_on.exy:.3f}")

aug_ell_off = DPAugmentor(add_ellipticity=True, ellipticity_scale=0.12,
                           add_ellipticity_to_label=False, rng=5)
dp_ell_off, label_ell_off = aug_ell_off.augment(dp, label=label_1ch)

show_2d([dp_ell_on, label_ell_on, dp_ell_off, label_ell_off],
        title=["DP (ell→label=True)", "Label (distorted)",
               "DP (ell→label=False)", "Label (undistorted)"],
        upper_quantile=0.99, cbar=True)

## 2. Signal Degradation

These augmentations simulate physical noise sources and detector artifacts. They are applied after the geometric transforms.

### 2a. Inelastic Background (`add_bkg`)

Adds a plasmon scattering background with form factor `1 / (q² + q₀²)`, optionally convolved with the probe function.

- `bkg_weight`: fraction of total intensity contributed by the background  
- `bkg_q`: `q₀` parameter controlling the fall-off; smaller → more peaked (centrally concentrated)  
- `probe`: if provided, the background is convolved with the probe, spreading it with the probe shape  

When `add_ellipticity=True` the form factor is evaluated in elliptically-transformed q-space so the background matches the distorted pattern geometry.

In [ ]:
aug_bkg_np = DPAugmentor(add_bkg=True, bkg_weight=0.15, bkg_q=0.05, rng=7)
dp_bkg_np = aug_bkg_np.augment(dp)                          # no probe

aug_bkg_p = DPAugmentor(add_bkg=True, bkg_weight=0.15, bkg_q=0.05, rng=7)
dp_bkg_p = aug_bkg_p.augment(dp, probe=probe)               # with probe

aug_bkg_ell = DPAugmentor(add_bkg=True, bkg_weight=0.15, bkg_q=0.05,
                           add_ellipticity=True, ellipticity_scale=0.12, rng=8)
dp_bkg_ell = aug_bkg_ell.augment(dp, probe=probe)           # ellipticity-aware
print(f"Ellipticity: exx={aug_bkg_ell.exx:.3f}, eyy={aug_bkg_ell.eyy:.3f}, exy={aug_bkg_ell.exy:.3f}")

show_2d([dp, dp_bkg_np, dp_bkg_p, dp_bkg_ell],
        title=["Original", "Background (no probe)", "Background + probe",
               "Background + probe + ellip."],
        upper_quantile=0.99, cbar=True)

### 2b. Shot Noise (`add_shot`)

Applies Poisson noise scaled by `e_dose` (electrons per diffraction pattern). The image is normalised to a probability distribution, sampled with `e_dose` electrons, and rescaled. Lower dose → more granular noise.

Typical ranges:  
- Fast scan / low dose: `1e3–1e4`  
- Standard: `1e5–1e6`  
- High dose: `1e7+`

In [ ]:
aug_s1 = DPAugmentor(add_shot=True, e_dose=1e7, rng=9)
dp_s1 = aug_s1.augment(dp)

aug_s2 = DPAugmentor(add_shot=True, e_dose=1e5, rng=9)
dp_s2 = aug_s2.augment(dp)

aug_s3 = DPAugmentor(add_shot=True, e_dose=1e3, rng=9)
dp_s3 = aug_s3.augment(dp)

show_2d([dp, dp_s1, dp_s2, dp_s3],
        title=["Original", "e_dose=1e7", "e_dose=1e5", "e_dose=1e3"],
        upper_quantile=0.99, cbar=True)

### 2c. Gaussian Noise (`add_gaussian_noise`)

Adds clipped Gaussian noise (values clipped to ≥ 0), simulating detector readout or dark-current noise.

- `gaussian_noise_mu`, `gaussian_noise_std`: when `add_shot=True`, these are multiplied by `e_dose` to scale with the signal level; otherwise they are absolute intensity values.

**Coupled profiles via `gaussian_noise_profiles`:** For more realistic noise modeling, profiles allow correlated `(mu, std)` pairs drawn together with specified weights. Each profile is a dict with `"weight"`, `"mu"`, and `"std"` keys (values can be scalars or `[min, max]` ranges). This overrides `gaussian_noise_mu/std`.

In [ ]:
aug_gn = DPAugmentor(add_gaussian_noise=True,
                     gaussian_noise_mu=0.0,
                     gaussian_noise_std=[1e-3, 5e-3],
                     rng=10)
dp_gn = aug_gn.augment(dp)
print(f"Gaussian noise: mu={aug_gn.gaussian_noise_mu:.2e}, std={aug_gn.gaussian_noise_std:.2e}")

# Coupled profiles: 70% clean, 30% noisy
aug_prof = DPAugmentor(
    add_gaussian_noise=True,
    gaussian_noise_profiles=[
        {"weight": 0.7, "mu": 0.0,         "std": 1e-4},
        {"weight": 0.3, "mu": [0.0, 5e-4], "std": [1e-3, 5e-3]},
    ],
    rng=11,
)
dp_prof = aug_prof.augment(dp)
print(f"Profiles drew:  mu={aug_prof.gaussian_noise_mu:.2e}, std={aug_prof.gaussian_noise_std:.2e}")

show_2d([dp, dp_gn, dp_prof],
        title=["Original", "Gaussian noise (range)", "Gaussian noise (profiles)"],
        upper_quantile=0.99, cbar=True)

### 2d. Salt & Pepper Noise (`add_salt_and_pepper`)

Sets a random fraction `salt_and_pepper` of pixels to either the image maximum (salt) or zero (pepper), simulating dead or hot detector pixels.

### 2e. Gaussian Blur (`add_blur`)

Applies isotropic Gaussian blur with standard deviation `blur_sigma` pixels. Simulates a defocused or aberrated detector point-spread function.

In [ ]:
aug_sp = DPAugmentor(add_salt_and_pepper=True, salt_and_pepper=5e-3, rng=12)
dp_sp = aug_sp.augment(dp)

aug_blur = DPAugmentor(add_blur=True, blur_sigma=[0.5, 2.5], rng=13)
dp_blur = aug_blur.augment(dp)
print(f"Blur sigma drawn: {aug_blur.blur_sigma:.2f} px")

aug_spblur = DPAugmentor(
    add_salt_and_pepper=True, salt_and_pepper=[0, 3e-3],
    add_blur=True, blur_sigma=[0, 1.5],
    rng=14,
)
dp_spblur = aug_spblur.augment(dp)

show_2d([dp, dp_sp, dp_blur, dp_spblur],
        title=["Original", "Salt & pepper", "Blur", "Salt & pepper + blur"],
        upper_quantile=0.99, cbar=True)

## 3. Labels

Labels receive the same geometric transforms (flip/rotation, shift, scale, ellipticity) but **not** noise augmentations (shot noise, blur, etc.), since labels represent ground-truth geometry.

Labels must be `float32` arrays. They can be single-channel `(H, W)` or multichannel `(C, H, W)`.

### 3a. Single-channel label

A single `(H, W)` label is warped identically to the DP. Toggle `add_ellipticity_to_label` to control whether the label also receives the elliptical warp (covered in Section 1d; see that section for a side-by-side comparison).

### 3b. Multichannel labels & `apply_background_to_label`

Pass a `(C, H, W)` label array to transform each channel independently. `apply_background_to_label` is a list of booleans (one per channel) that controls whether the inelastic background is also applied to that channel. This is useful when one channel is a copy of the DP itself (e.g. for autoencoder training) and should receive background degradation.

In [ ]:
# Multichannel label: ch0 = DP copy (should get background), ch1 = binary mask
label_mc = np.stack([
    dp.copy(),
    (dp > dp.mean()).astype(np.float32),
]).astype(np.float32)
print(f"Multichannel label shape: {label_mc.shape}")

aug_mc = DPAugmentor(
    add_bkg=True, bkg_weight=0.15, bkg_q=0.05,
    apply_background_to_label=[True, False],     # background only on ch0
    add_ellipticity=True, ellipticity_scale=0.1,
    add_ellipticity_to_label=True,
    add_flipshift=True, free_rotation=False,
    rng=16,
)
dp_mc, label_mc_aug = aug_mc.augment(dp, probe=probe, label=label_mc)

show_2d(
    [dp_mc, label_mc_aug[0], label_mc_aug[1]],
    title=["Augmented DP", "Label ch0 (bkg applied)", "Label ch1 (no bkg)"],
    upper_quantile=0.99, cbar=True,
)

## 4. Weighted Parameter Mixtures

Every numeric parameter can be given as a weighted mixture of ranges instead of a single `[min, max]`. This allows sampling from non-uniform or multi-modal distributions.

```python
# Syntax: list of dicts with "weight" and "value" keys
e_dose=[
    {"weight": 0.7, "value": [1e5, 1e6]},   # 70%: moderate dose
    {"weight": 0.3, "value": [1e3, 1e4]},   # 30%: low dose
]
```

Weights are normalised automatically; only their ratios matter. For Gaussian noise the `gaussian_noise_profiles` parameter couples `mu` and `std` together (Section 2c).

In [ ]:
import matplotlib.pyplot as plt

aug_mix = DPAugmentor(
    add_shot=True,
    e_dose=[
        {"weight": 0.7, "value": [1e5, 1e6]},
        {"weight": 0.3, "value": [1e3, 1e4]},
    ],
    add_bkg=True,
    bkg_weight=[
        {"weight": 0.5, "value": 0.0},
        {"weight": 0.5, "value": [0.05, 0.2]},
    ],
    bkg_q=[0.03, 0.1],
    rng=17,
)

# Sample 500 parameter draws to visualise the distributions
doses, bkg_weights = [], []
for _ in range(500):
    aug_mix.generate_params()
    doses.append(aug_mix.e_dose)
    bkg_weights.append(aug_mix.bkg_weight)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(np.log10(doses), bins=30, edgecolor="k", linewidth=0.5)
axes[0].set(xlabel="log10(e_dose)", title="Sampled electron doses (n=500)")
axes[1].hist(bkg_weights, bins=30, edgecolor="k", linewidth=0.5)
axes[1].set(xlabel="bkg_weight", title="Sampled background weights (n=500)")
plt.tight_layout()
plt.show()

## 5. Stack Augmentation

`augment()` accepts an `(N, H, W)` stack and augments each image independently with freshly sampled parameters. The `probe` and `label` stacks must match the batch dimension `N`.

In [ ]:
aug_stk = DPAugmentor(
    add_flipshift=True, free_rotation=False,
    add_shift=True, xshift=[0, 10], yshift=[0, 10],
    add_ellipticity=True, ellipticity_scale=[0, 0.1],
    add_shot=True, e_dose=[1e4, 1e6],
    rng=18,
)

label_stack = dp_stack.copy()   # autoencoder-style: label = input
aug_out, label_out = aug_stk.augment(dp_stack, label=label_stack)
print(f"Input: {dp_stack.shape}  →  Augmented: {aug_out.shape}")

n_show = min(4, len(dp_stack))
show_2d(
    [[dp_stack[i] for i in range(n_show)], [aug_out[i] for i in range(n_show)]],
    title=[["Original" for _ in range(n_show)], ["Augmented" for _ in range(n_show)]],
    upper_quantile=0.99, cbar=True,
)

## 6. Reproducibility & Logging

### Reproducibility

Pass `rng=<int>` to fix the seed. The same seed produces byte-identical outputs across runs. The seed is stored in `augmentor._rng_seed`.

### Logging

Pass `log_file="path/to/log.csv"` to record every augmented image's parameters in a CSV. One row per image; columns include all numeric parameters and the seed. Useful for debugging or auditing training data.

In [ ]:
import tempfile, os

# Same seed → identical output
aug_a = DPAugmentor(add_shift=True, xshift=[0, 20], yshift=[0, 20], rng=42)
aug_b = DPAugmentor(add_shift=True, xshift=[0, 20], yshift=[0, 20], rng=42)
dp_a, dp_b = aug_a.augment(dp), aug_b.augment(dp)
print(f"Identical outputs with same seed: {np.allclose(dp_a, dp_b)}")

# Logging
with tempfile.NamedTemporaryFile(suffix=".csv", delete=False, mode="w") as f:
    log_path = f.name

aug_log = DPAugmentor(
    add_shift=True, xshift=[0, 15], yshift=[0, 15],
    add_ellipticity=True, ellipticity_scale=[0, 0.1],
    add_shot=True, e_dose=[1e4, 1e6],
    log_file=log_path, rng=99,
)
for _ in range(5):
    aug_log.augment(dp)

import pandas as pd
log_df = pd.read_csv(log_path)
print(f"\nLog ({log_path}):")
print(log_df[["xshift", "yshift", "exx", "eyy", "exy", "e_dose"]].to_string())
os.unlink(log_path)

## 7. Kitchen Sink — Full Augmentation Pipeline

A realistic configuration combining all augmentations. Use this as a starting template and tune individual ranges for your dataset.

In [ ]:
label_1ch = (dp > dp.mean()).astype(np.float32)

aug_full = DPAugmentor(
    # Geometric
    add_flipshift=True, free_rotation=False,          # 90° only: preserves 4-fold symmetry
    add_shift=True,       xshift=[0, 10],  yshift=[0, 10],
    add_scale=True,       scale_factor=[0.95, 1.05],
    add_ellipticity=True, ellipticity_scale=[0, 0.12],
    add_ellipticity_to_label=True,
    # Background
    add_bkg=True,
    bkg_weight=[0.01, 0.08],
    bkg_q=[0.02, 0.08],
    apply_background_to_label=None,
    # Shot noise (bimodal dose: mostly high, occasionally low)
    add_shot=True,
    e_dose=[
        {"weight": 0.6, "value": [1e5, 1e6]},
        {"weight": 0.4, "value": [1e3, 1e5]},
    ],
    # Detector noise
    add_gaussian_noise=True,
    gaussian_noise_mu=0.0,
    gaussian_noise_std=[1e-5, 1e-4],
    add_blur=True,            blur_sigma=[0, 1.0],
    add_salt_and_pepper=True, salt_and_pepper=[0, 2e-4],
    rng=0,
)
aug_full.print_params()

dp_full, label_full = aug_full.augment(dp, probe=probe, label=label_1ch)

show_2d([dp, dp_full, label_1ch, label_full],
        title=["Original DP", "Fully augmented DP",
               "Original label", "Augmented label"],
        upper_quantile=0.99, cbar=True)